---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: AI Engineering

### 📋 **Topic**: Preprocessing

🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---

## Welcome!

Before data can be used by an AI system, we may choose to **preprocess** it: to convert it to an easy-to-use format, chunk it into smaller pieces, or to enrich it with metadata. Today's class will cover some of these strategies:

1. **Ingestion**: we will convert raw data, such as HTML, Word, PDFs, and Audio, to Markdown.
2. **Chunking**: we will explore strategies for splitting documents into retrieval-ready pieces
3. **Enrichment**: we will explore strategies for adding more information to chunks that can help us in the retrieval stage.

Preprocessing is no rocket science, but it is often one of the most consequential things you can improve in an AI system pipeline. I've seen AI systems go 70% to 90% recall by improving their preprocessing.

In [ ]:
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

DATA = Path("../data") # read the data from here
TEMP = Path("../temp") # save the converted data here
DATA.mkdir(exist_ok=True)
TEMP.mkdir(exist_ok=True)


# 1. Converting to text

---



## 1.1 HTML to Markdown

There are several libraries that can do that -- I use **markdownify** here. This is a simple library that converts HTML tags to their Markdown equivalents:

Here is an example of a real [page](https://www.fordham.edu/gabelli-school-of-business/academic-programs-and-admissions/graduate-programs/) from Fordham's website:

In [ ]:
html_path = DATA / "preprocessing-example.html"
html_content = html_path.read_text()
print(f"File size: {len(html_content):,} characters")
print()
print(html_content)

77,000 characters of tags, scripts, styles, and navigation — and buried somewhere in there is the actual content. Let's convert it.

In [ ]:
from markdownify import markdownify

# Convert the entire HTML to Markdown
full_markdown = markdownify(html_content, heading_style="ATX", strip=["img", "script", "style"])

# That's a LOT of output — let's just peek at the first 1000 chars
print(f"Converted: {len(full_markdown):,} characters")
print()
print(full_markdown)

Markdownify converted everything — including all the navigation, footer, and boilerplate. 
- That's not useful for a RAG pipeline. We need to extract just the main content.
- You're probably better off with `BeautifulSoup` so that you can more "surgically" extract the `<main>` or content `<div>` (what we'll do here)
- You could also use `trafilatura` — a library designed to extract article content from web pages automatically

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html_content, "html.parser")

# Real websites use different containers — try <main>, then common class names
main_content = soup.find("main") or soup.find("div", class_="content")

if main_content:
    clean_markdown = markdownify(str(main_content), heading_style="ATX", strip=["img"])
    print(f"Extracted main content: {len(clean_markdown):,} characters")
    print()
    print(clean_markdown[:])
else:
    print("No <main> tag found — using full conversion")
    clean_markdown = full_markdown

In [ ]:
# Save the converted Markdown
output_path = TEMP / "preprocessing-example-html.md"
output_path.write_text(clean_markdown)
print(f"Saved to {output_path}")

## 1.2 Word Documents (DOCX)

Let's convert our example — a course syllabus.

In [ ]:
from docx import Document

docx_path = DATA / "preprocessing-example.docx"
doc = Document(docx_path)

print(f"Paragraphs: {len(doc.paragraphs)}")
print(f"Tables:     {len(doc.tables)}")
print()

# The style name tells us the document structure
for i, para in enumerate(doc.paragraphs[:12]):
    print(f"  [{i:2d}] style={para.style.name:20s} | {para.text[:70]}")

Notice the `style` column — Word uses named styles like `Heading 1`, `Heading 2`, `Normal`, `List Bullet` to indicate document structure. This can help us convert it to Markdown
-  The key trick is iterating through `doc.element.body` (the raw XML children) rather than `doc.paragraphs` and `doc.tables` separately. This preserves the actual document order — tables appear exactly where they belong.

In [ ]:
from docx.text.paragraph import Paragraph
from docx.table import Table


def docx_to_markdown(doc: Document) -> str:
    """Convert a python-docx Document to Markdown, preserving paragraph and table order."""
    lines = []

    # Iterate the XML body in document order (paragraphs + tables interleaved)
    for element in doc.element.body:
        tag = element.tag.split("}")[-1]  # Strip XML namespace

        if tag == "p":  # Paragraph
            para = Paragraph(element, doc)
            text = para.text.strip()
            if not text:
                continue

            style = para.style.name

            if style.startswith("Heading"):
                level = int(style.split()[-1])
                lines.append(f"{'#' * level} {text}")
            elif style == "List Bullet":
                lines.append(f"- {text}")
            elif style == "List Number":
                lines.append(f"1. {text}")
            else:
                lines.append(text)
            lines.append("")

        elif tag == "tbl":  # Table
            table = Table(element, doc)
            rows = []
            for row in table.rows:
                cells = [cell.text.strip() for cell in row.cells]
                rows.append("| " + " | ".join(cells) + " |")
            if rows:
                header_sep = "| " + " | ".join(["---"] * len(table.rows[0].cells)) + " |"
                rows.insert(1, header_sep)
                lines.append("\n".join(rows))
                lines.append("")

    return "\n".join(lines)

In [ ]:
docx_markdown = docx_to_markdown(doc)
print(docx_markdown)

In [ ]:
output_path = TEMP / "preprocessing-example-docx.md"
output_path.write_text(docx_markdown)
print(f"Saved to {output_path}")

## 1.3 PDF Documents

PDFs are the hardest common format to convert. Here's why:

| HTML / DOCX | PDF |
|---|---|
| `<h1>Title</h1>` — explicit heading | "Draw 'Title' at position (72, 80) in 24pt font" |
| `<table>` — explicit table structure | Lines and text placed to *look like* a table |
| `<p>` — explicit paragraph | Characters placed sequentially until a position jump |

A PDF doesn't know it has "headings" or "tables" — it just knows where to put ink on a page. 
- This is why some of the best extractors use OCR or vLLMs

We're going to use as our example a real, 5-page document: the [Washington State Executive Order 24-01 on Artificial Intelligence](https://governor.wa.gov/sites/default/files/exe_order/24-01%20-%20Artificial%20Intelligence%20%28tmp%29.pdf)

We'll try two different tools and compare the results.

**pymupdf4llm** is a lightweight wrapper around PyMuPDF that outputs Markdown. It's fast and works well for most native (non-scanned) PDFs. It infers headings from font size and weight.

In [ ]:
import pymupdf4llm

pdf_path = DATA / "preprocessing-example.pdf"

# Convert PDF to Markdown — one line!
pymupdf_markdown = pymupdf4llm.to_markdown(str(pdf_path))

# Show the first 2000 characters
print(f"Total output: {len(pymupdf_markdown):,} characters")
print()
print(pymupdf_markdown[:])

In [ ]:
# pymupdf4llm also supports per-page chunking
pages = pymupdf4llm.to_markdown(str(pdf_path), page_chunks=True)

print(f"Document has {len(pages)} pages\n")
for i, page in enumerate(pages):
    preview = page["text"][:].replace("\n", " ")
    print(f"Page {i + 1} ({len(page['text']):,} chars): {preview}...")

In [ ]:
output_path = TEMP / "preprocessing-example-pdf-pymupdf.md"
output_path.write_text(pymupdf_markdown)
print(f"Saved to {output_path}")

**Docling** is heavier than pymupdf4llm (it downloads and runs ML models), but it understands document layout at a deeper level — especially useful for complex PDFs with multi-column layouts, scanned pages, or intricate tables.

In [ ]:
from docling.document_converter import DocumentConverter

docling_output_path = TEMP / "preprocessing-example-pdf-docling.md"

converter = DocumentConverter()
result = converter.convert(str(pdf_path))
docling_markdown = result.document.export_to_markdown()
docling_output_path.write_text(docling_markdown)

print(f"Total output: {len(docling_markdown):,} characters")
print()
print(docling_markdown)

Let's compare the two outputs on the same document.

In [ ]:
print(f"{'':30s} {'pymupdf4llm':>12s}  {'docling':>12s}")
print(f"{'Characters':30s} {len(pymupdf_markdown):>12,d}  {len(docling_markdown):>12,d}")
print(f"{'Lines':30s} {len(pymupdf_markdown.splitlines()):>12,d}  {len(docling_markdown.splitlines()):>12,d}")

pymupdf_headings = [l for l in pymupdf_markdown.splitlines() if l.startswith("#")]
docling_headings = [l for l in docling_markdown.splitlines() if l.startswith("#")]
print(f"{'Headings detected':30s} {len(pymupdf_headings):>12d}  {len(docling_headings):>12d}")

print(f"\npymupdf4llm headings: {pymupdf_headings[:5]}")
print(f"docling headings:     {docling_headings[:5]}")

The PDF parsing space has exploded after 2024, driven by RAG and LLM demand:

| Tool | Approach | Best For | License |
|---|---|---|---|
| **pymupdf4llm** | Rule-based (font heuristics) | Fast general-purpose conversion | AGPL |
| **Docling** (IBM) | AI models (layout + tables + OCR) | Complex layouts, scanned docs | MIT |
| **MinerU** (OpenDataLab) | AI models (DocLayout-YOLO) | Multi-language, CJK documents | AGPL |
| **Marker** (Datalab) | AI models (Surya OCR) | Fast batch processing, equations | GPL |
| **pdfplumber** | Rule-based (geometric analysis) | Precise table extraction | MIT |
| **pypdf** | Rule-based (pure Python) | Simple text extraction, zero deps | BSD |

**Rule of thumb**: Start with pymupdf4llm. If the output isn't good enough (bad tables, missed structure, scanned pages), try Docling or Marker.

## 1.4 Audio Transcription

Audio is increasingly important for RAG — meeting recordings, lectures, podcasts, customer calls. OpenAI's transcription API makes this easy: 3 lines of code, $0.003/minute.

Our example is a LibriVox recording of Abraham Lincoln's Gettysburg Address (public domain, ~2.5 minutes).

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

from openai import OpenAI

client = OpenAI()

audio_path = DATA / "preprocessing-example.mp3"
print(f"Audio file: {audio_path.name} ({audio_path.stat().st_size / 1024:.0f} KB)")

In [ ]:
# Transcribe with OpenAI's API — that's it!
with open(audio_path, "rb") as f:
    transcript = client.audio.transcriptions.create(
        model="gpt-4o-mini-transcribe",
        file=f,
    )

print(transcript.text)

In [ ]:
# Save the transcript
output_path = TEMP / "preprocessing-example-audio.md"
output_path.write_text(transcript.text)
print(f"Saved to {output_path} ({len(transcript.text):,} characters)")

### Audio Transcription Options

| Tool | Cost | Quality | Notes |
|---|---|---|---|
| **OpenAI API** (`gpt-4o-mini-transcribe`) | $0.003/min | Excellent | 3 lines of code, best accuracy |
| **OpenAI API** (`whisper-1`) | $0.006/min | Great | Original Whisper model via API |
| **faster-whisper** (local) | Free | Good | Runs locally, needs GPU for speed |
| **OpenAI Whisper** (local) | Free | Good | The original open-source model |

For video files, extract the audio track first (e.g., with `ffmpeg` or `moviepy`), then transcribe.

## 1.5 Paid Services

There are also commercial services that handle document conversion at scale:

| Service | What It Does | Pricing |
|---|---|---|
| **LlamaParse** (LlamaIndex) | Cloud PDF/DOCX parsing optimized for LLMs. Highest accuracy on complex documents (tables, forms, multi-column). | Free: 1,000 pages/day. Paid plans from $3/1k pages. |
| **Unstructured Platform** | Cloud ETL for documents. Supports 64+ file types. Semantic element classification (Title, NarrativeText, Table, etc.) | Free tier available. Enterprise pricing. |

When you have many complex documents, need guaranteed uptime, or the open-source tools aren't producing good enough output for your use case, you may want to use a paid service. But for prototyping and learning, use an open-source tool.


# 2. Chunking

---

We saw that most documents are too long to embed as a single unit or to stuff into an LLM's context window. We need to split them into smaller pieces that are:
- small enough to be specific (so retrieval is precise)
- large enough to be self-contained (so the LLM has enough context)
- clean at boundaries (so we don't cut mid-sentence or mid-thought)

In [ ]:
import tiktoken
import numpy as np

# Load the preprocessed documents from Part 1
pdf_text = (TEMP / "preprocessing-example-pdf-pymupdf.md").read_text()
html_text = (TEMP / "preprocessing-example-html.md").read_text()
docx_text = (TEMP / "preprocessing-example-docx.md").read_text()

print(f"PDF  (WA Executive Order): {len(pdf_text):,} chars, {len(pdf_text.splitlines()):,} lines")
print(f"HTML (Fordham programs):   {len(html_text):,} chars, {len(html_text.splitlines()):,} lines")
print(f"DOCX (Course syllabus):    {len(docx_text):,} chars, {len(docx_text.splitlines()):,} lines")

In [ ]:
# Initialize the tokenizer — cl100k_base is used by text-embedding-3-* and GPT-4
# (Note: GPT-4o uses o200k_base, but cl100k_base is the right choice for counting embedding tokens)
enc = tiktoken.get_encoding("cl100k_base")


def count_tokens(text: str) -> int:
    """Count the number of tokens in a text string."""
    return len(enc.encode(text))


def show_chunks(chunks: list[str], max_preview: int = 80) -> None:
    """Display a summary table of chunks with index, token count, and preview."""
    print(f"{'#':>3s}  {'Tokens':>6s}  {'Chars':>6s}  Preview")
    print("-" * 100)
    for i, chunk in enumerate(chunks):
        tokens = count_tokens(chunk)
        preview = chunk.replace("\n", " ")[:max_preview]
        print(f"{i:3d}  {tokens:6d}  {len(chunk):6d}  {preview}...")
    total_tokens = sum(count_tokens(c) for c in chunks)
    print("-" * 100)
    print(f"Total: {len(chunks)} chunks, {total_tokens:,} tokens, {sum(len(c) for c in chunks):,} chars")

## 2.1 Fixed-Size Character Chunking

The simplest possible approach: split the text into equal-sized pieces by character count.
- This is the baseline. It's fast, predictable, and easy to implement. 
- It also has an obvious flaw — it cuts through words, sentences, and even paragraphs without regard for meaning.

In [ ]:
def chunk_by_chars(text: str, chunk_size: int = 500) -> list[str]:
    """Split text into fixed-size character chunks."""
    return [text[i : i + chunk_size] for i in range(0, len(text), chunk_size)]


# Try it on the executive order
chunks = chunk_by_chars(pdf_text, chunk_size=500)
show_chunks(chunks)

In [ ]:
# Look at a boundary between two chunks — where does it cut?
boundary_idx = 3
print(f"=== END of chunk {boundary_idx} ===")
print(repr(chunks[boundary_idx][-60:]))
print()
print(f"=== START of chunk {boundary_idx + 1} ===")
print(repr(chunks[boundary_idx + 1][:60]))

Notice the cut happens in the middle of a sentence. If a user asks a question whose answer spans that boundary, neither chunk contains the complete answer. The retrieval system would return a fragment, and the LLM would have to work with an incomplete thought.
- This is the fundamental tension of chunking: **smaller chunks are more precise for retrieval, but risk splitting information across boundaries**.
- Fixed-size character chunking is rarely used in production, but it's the right starting point for understanding the problem. Every other strategy is an attempt to make boundaries smarter.

## 2.2 Overlapping Chunks (Sliding Window)

The previous approach splits boundaries in two. If the answer to a user's question straddles a boundary, neither chunk contains the full answer. 

One fix is to add some **overlap**. Each chunk includes some text from the end of the previous chunk, creating redundancy at boundaries.

> 📚 **TERM: Chunk Overlap**
> The number of characters or tokens that adjacent chunks share. If chunk_size=200 and overlap=50, each chunk starts 150 tokens after the previous one, and the last 50 tokens of chunk N are also the first 50 tokens of chunk N+1.

Overlap increases the total number of text we need to store, but makes it more likely that no information falls through the cracks.

In [ ]:
def chunk_by_tokens_overlap(text: str, chunk_size: int = 200, overlap: int = 50) -> list[str]:
    """Split text into token chunks with overlap between adjacent chunks."""
    tokens = enc.encode(text)
    step = chunk_size - overlap
    chunks = []
    for i in range(0, len(tokens), step):
        chunk_tokens = tokens[i : i + chunk_size]
        if not chunk_tokens:
            break
        chunks.append(enc.decode(chunk_tokens))
    return chunks


# Compare: no overlap vs 50-token overlap
chunks_with_overlap = chunk_by_tokens_overlap(pdf_text, chunk_size=200, overlap=50)


print(f"With 50-token overlap: {len(chunks_with_overlap)} chunks")

In [ ]:
# Show that the end of chunk 2 matches the start of chunk 3
i = 2
end_of_prev = chunks_with_overlap[i][-200:]
start_of_next = chunks_with_overlap[i + 1][:200]

print(f"=== END of chunk {i} (last 200 chars) ===")
print(end_of_prev)
print()
print(f"=== START of chunk {i + 1} (first 200 chars) ===")
print(start_of_next)

**How much overlap?**

Research and practice suggest about a **10-20% overlap** as the sweet spot. 
- This is of course subject to change. 
- Overlap is almost always worth the cost. The storage overhead is small (10-25%), and the retrieval improvement at boundaries can be significant. Default to 10-15% overlap unless you have a reason not to.

## 2.3 Recursive Character Splitting

The previous approaches all ignore the document's natural structure. But text already has built-in boundaries: paragraphs, sentences, line breaks. **Recursive character splitting** takes advantage of this, and is one of the most popular chunking strategy in production RAG systems. The idea:

1. Try to split on the largest structural boundary: double newlines (`\n\n` = paragraphs)
2. If any piece is still too large, split on single newlines (`\n`)
3. If still too large, split on sentences (`. `)
4. If still too large, split on spaces (` `)
5. Last resort: split on characters

Each level preserves more context than the one below it. The algorithm tries to keep the largest meaningful units intact.
- This was popularized by LangChain's `RecursiveCharacterTextSplitter`.

In [ ]:
def recursive_split(
    text: str,
    chunk_size: int = 200,
    overlap: int = 40,
    separators: list[str] | None = None,
) -> list[str]:
    """
    Recursively split text using a hierarchy of separators.

    Tries each separator in order. If a piece is still too large after splitting,
    recurse with the next separator. Includes overlap between chunks.
    """
    if separators is None:
        separators = ["\n\n", "\n", ". ", " ", ""]

    # Base case: text fits in one chunk
    if count_tokens(text) <= chunk_size:
        return [text.strip()] if text.strip() else []

    # Try each separator
    for sep in separators:
        if sep == "":
            # Last resort: token-level split
            return chunk_by_tokens_overlap(text, chunk_size, overlap)

        pieces = text.split(sep)
        if len(pieces) == 1:
            continue  # This separator doesn't split the text; try the next one

        # Merge pieces into chunks that fit within chunk_size
        chunks = []
        current = pieces[0]

        for piece in pieces[1:]:
            candidate = current + sep + piece
            if count_tokens(candidate) <= chunk_size:
                current = candidate
            else:
                # Current chunk is full — save it and start a new one
                if current.strip():
                    chunks.append(current.strip())
                # Start new chunk with overlap from the end of the previous
                if overlap > 0 and chunks:
                    prev_tokens = enc.encode(chunks[-1])
                    overlap_text = enc.decode(prev_tokens[-overlap:])
                    current = overlap_text + sep + piece
                else:
                    current = piece

        if current.strip():
            chunks.append(current.strip())

        # Recursively split any chunks that are still too large
        final_chunks = []
        remaining_seps = separators[separators.index(sep) + 1 :]
        for chunk in chunks:
            if count_tokens(chunk) > chunk_size:
                final_chunks.extend(
                    recursive_split(chunk, chunk_size, overlap, remaining_seps)
                )
            else:
                final_chunks.append(chunk)

        return final_chunks

    return [text.strip()] if text.strip() else []

In [ ]:
chunks = recursive_split(pdf_text, chunk_size=200, overlap=40)
show_chunks(chunks)

Recursive splitting generally produces chunks that end at natural boundaries — paragraph breaks or sentence endings.  — while the token-based splitter cuts mid-sentence.
- Chunks are more **semantically coherent** (they contain complete thoughts)
- It's **deterministic and fast** (no ML models needed)
- It handles **any document** without requiring specific structure

It's a good idea to start with with recursive splitting at 400-512 tokens with 10-15% overlap. Some research has show that this splitting achieves good performance (see Chroma's research).

## 2.4 Semantic Chunking

All previous strategies use surface-level signals to help us to decide where to split. 
- character counts, token counts, newlines, headings. 

None of them actually understand the content of the text. Semantic chunking moves towards this direction, and uses **embeddings** to detect topic shifts. The idea is the following:
1. Split the document into small units (sentences)
2. Embed each sentence
3. Compare the similarity of adjacent sentences
4. Split where the similarity drops — that's where the topic changes

This is the most sophisticated chunking strategy that doesn't require an LLM.

In [ ]:
import re


def split_sentences(text: str) -> list[str]:
    """Split text into sentences using a simple regex."""
    sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z])", text)
    result = []
    for s in sentences:
        parts = s.split("\n\n")
        result.extend(p.strip() for p in parts if p.strip())
    return result


def chunk_by_tokens(text: str, chunk_size: int = 200) -> list[str]:
    """Split text into chunks of approximately chunk_size tokens."""
    tokens = enc.encode(text)
    chunks = []
    for i in range(0, len(tokens), chunk_size):
        chunk_tokens = tokens[i : i + chunk_size]
        chunks.append(enc.decode(chunk_tokens))
    return chunks


def chunk_by_sentences(
    text: str, chunk_size: int = 200, overlap_sentences: int = 1
) -> list[str]:
    """Group sentences into chunks that fit within chunk_size tokens."""
    sentences = split_sentences(text)
    chunks = []
    current_sentences = []
    current_tokens = 0

    for sentence in sentences:
        sentence_tokens = count_tokens(sentence)
        if current_tokens + sentence_tokens > chunk_size and current_sentences:
            chunks.append(" ".join(current_sentences))
            current_sentences = (
                current_sentences[-overlap_sentences:] if overlap_sentences > 0 else []
            )
            current_tokens = sum(count_tokens(s) for s in current_sentences)
        current_sentences.append(sentence)
        current_tokens += sentence_tokens

    if current_sentences:
        chunks.append(" ".join(current_sentences))
    return chunks

In [ ]:
from helpers import get_local_model


def chunk_semantic(
    text: str,
    threshold: float = 0.5,
    model_name: str = "all-MiniLM-L6-v2",
) -> list[str]:
    """
    Split text into chunks based on semantic similarity between adjacent sentences.

    1. Split into sentences
    2. Embed each sentence
    3. Compute cosine similarity between consecutive sentences
    4. Split where similarity drops below threshold
    """
    sentences = split_sentences(text)
    if len(sentences) <= 1:
        return sentences

    # Embed all sentences
    model = get_local_model(model_name)
    embeddings = model.encode(sentences, convert_to_numpy=True)

    # Compute similarity between adjacent sentences
    similarities = []
    for i in range(len(embeddings) - 1):
        sim = np.dot(embeddings[i], embeddings[i + 1]) / (
            np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i + 1])
        )
        similarities.append(sim)

    # Find split points
    split_points = [0]
    for i, sim in enumerate(similarities):
        if sim < threshold:
            split_points.append(i + 1)
    split_points.append(len(sentences))

    # Build chunks
    chunks = []
    for i in range(len(split_points) - 1):
        chunk_sentences = sentences[split_points[i] : split_points[i + 1]]
        chunk_text = " ".join(chunk_sentences)
        chunks.append(chunk_text)

    return chunks

In [ ]:
chunks = chunk_semantic(pdf_text, threshold=0.5)
show_chunks(chunks)

In [ ]:
import matplotlib.pyplot as plt

# Compute inter-sentence similarities for visualization
sentences = split_sentences(pdf_text)
model = get_local_model("all-MiniLM-L6-v2")
embeddings = model.encode(sentences, convert_to_numpy=True)

similarities = []
for i in range(len(embeddings) - 1):
    sim = np.dot(embeddings[i], embeddings[i + 1]) / (
        np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i + 1])
    )
    similarities.append(sim)

plt.figure(figsize=(12, 4))
plt.plot(similarities, linewidth=1, color="steelblue")
plt.axhline(y=0.5, color="red", linestyle="--", label="Threshold (0.5)")
plt.xlabel("Sentence Index")
plt.ylabel("Cosine Similarity to Next Sentence")
plt.title("Inter-Sentence Similarity — WA Executive Order on AI")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

splits_below = sum(1 for s in similarities if s < 0.5)
print(f"Sentences: {len(sentences)}, Similarity drops below 0.5: {splits_below} times")

In [ ]:
# Try different thresholds and see the effect
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    chunks = chunk_semantic(pdf_text, threshold=threshold)
    total_tokens = sum(count_tokens(c) for c in chunks)
    avg_tokens = total_tokens // len(chunks) if chunks else 0
    print(f"Threshold {threshold:.1f}: {len(chunks):2d} chunks, avg {avg_tokens:3d} tokens/chunk")

At its best, semantic chunking:
- Can split  based on actual content shifts, not arbitrary boundaries
- Can adapt to the document's natural structure
- Can work on any document without special formatting or setup.

However, this power comes at the cost of lower speed (often does not matter much). More importantly
- It is sensitive to the threshold used— the right threshold varies by document and embedding model
- It is not always better -- some researchers have found that simple fixed-size chunking at 512 tokens often outperforms semantic chunking in retrieval benchmarks

In our executive order document, semantic chunking performed pretty poorly: dozens of tiny chunks averaging just ~30 tokens. The reason is that the document's repetitive "WHEREAS" clauses and formulaic legal structure caused the embedding model to see constant topic shifts at nearly every sentence boundary. This is a good reminder that no strategy works universally — semantic chunking excels on documents with clear, distinct topic sections, but struggles with repetitive or formulaic text.

## 2.5 Advanced Chunking Strategies

---

The strategies above cover what you'll implement in practice. But the field is evolving rapidly. Here are approaches worth knowing about — even if you won't implement them from scratch.

### 2.5.1 Late Chunking

Standard chunking embeds each chunk independently — the embedding for chunk 3 knows nothing about chunks 1, 2, or 4. **Late chunking** flips this:

1. Run the **entire document** through a long-context embedding model
2. The model produces a token-level embedding for every token, conditioned on the full document
3. **Then** split into chunks and pool the token embeddings within each chunk

The result: each chunk's embedding is informed by the full document context. A chunk about "the committee" benefits from knowing that the document is about AI governance in Washington state.

Providers: **Jina AI** (jina-embeddings-v3) and **Voyage AI** (voyage-context-3) support late chunking natively.

**Trade-off**: Requires long-context embedding models (8K+ tokens). Not supported by all embedding APIs.

### 2.5.2 Contextual Retrieval (Anthropic)

Instead of embedding chunks in isolation, **prepend a short context summary** to each chunk before embedding:

```
Original chunk: "The committee shall submit a report by September 2024."

With context: "This chunk is from Washington State Executive Order 24-01 on AI governance.
It describes a directive requiring WaTech to report on AI initiatives.
---
The committee shall submit a report by September 2024."
```

The context is generated by an LLM that sees the full document and the chunk. This costs one LLM call per chunk but dramatically improves retrieval — Anthropic reported a **49% reduction in retrieval failures** when combined with hybrid search.

**Trade-off**: Expensive at indexing time (one LLM call per chunk), but no cost at query time. The context is baked into the embedding.

### 2.5.3 Hierarchical Chunking (Small-to-Big)

Use **two levels of chunks**:
- **Child chunks** (small, ~128 tokens): Used for retrieval — precise matching
- **Parent chunks** (large, ~1024 tokens): Used for generation — rich context

At retrieval time, find the most relevant child chunk, then return its parent for the LLM to use. This gives you precise retrieval AND rich context.

```
Parent chunk (1024 tokens): [Full section about AI procurement guidelines...]
  |-- Child chunk 1 (128 tokens): [Paragraph about vendor certification...]
  |-- Child chunk 2 (128 tokens): [Paragraph about risk assessment...]
  |-- Child chunk 3 (128 tokens): [Paragraph about public posting...]
```

This is sometimes called "small-to-big retrieval" and is a common pattern in production RAG systems using frameworks like LlamaIndex.

### 2.5.4 LLM-Based (Agentic) Chunking

Let an LLM read the document and decide where to split. The LLM considers topic coherence, whether each chunk is self-contained, and optimal boundaries for downstream retrieval.

Chroma's research found this achieves **91.9% recall** — the best of any strategy — but it has a higher cost (one LLM call per boundary decision).

**Trade-off**: Best quality, but 100-1000x more expensive and slower than rule-based approaches. Practical only for high-value, low-volume document sets.

### 2.5.5 Page-Level Chunking

For PDFs specifically: just use the page boundaries that already exist. Each page becomes one chunk.

NVIDIA's 2024 RAG benchmark found that page-level chunking **won overall** with 0.648 accuracy and the lowest variance across different query types. This is surprising — it's the simplest possible approach for PDFs.

Why it works: PDF pages are already human-curated units of information. Authors organize content to fit pages. The page is a natural "chunk" that humans already designed.

**Trade-off**: Only works for PDFs. Pages can be very long (1000+ tokens) which may hurt retrieval precision. The `pymupdf4llm` library we used in Part 1 already supports `page_chunks=True` for this.

### 2.5.6 Chunking Libraries

We implemented everything from scratch to understand the concepts. In practice, there are mature libraries that handle the edge cases:

| Library | Key Feature | Notes |
|---|---|---|
| **LangChain** | `RecursiveCharacterTextSplitter` | Industry standard, many splitter types |
| **LlamaIndex** | `SentenceSplitter`, `SemanticSplitter` | Tight integration with their RAG framework |
| **Chonkie** | Lightweight, fast chunking | New library focused purely on chunking |
| **Unstructured** | Element-aware chunking | Splits by document elements (titles, paragraphs, tables) |

These libraries handle edge cases we skipped (e.g., robust sentence detection, HTML tag preservation, table handling). Once you understand the concepts, using a library is the right choice for production.

# 3. Enrichment

---

When we extract a chunk from a document, it loses context. 
- For example, a chunk that says *"The committee shall submit a report by September 2024"* doesn't tell you which committee, what report, or what document this came from. An embedding of this chunk will be generic — and retrieval will suffer.

**Enrichment** is the process with which we add information to each chunk before embedding it:
1. Contextual enrichment — prepend a short context summary
2. Hypothetical question generation — generate questions the chunk can answer
3. Metadata extraction — extract topics, entities, dates for structured filtering

All three use LLM calls at **indexing time** (not query time). The cost is paid once, and every future query benefits.

In [ ]:
import litellm
import asyncio
import json
import pandas as pd
from pydantic import BaseModel, Field


chunks = recursive_split(pdf_text, chunk_size=256, overlap=38)

print(f"Created {len(chunks)} chunks from the executive order ({count_tokens(pdf_text):,} tokens)\n")
for i, chunk in enumerate(chunks[:3]):
    preview = chunk.replace("\n", " ")[:100]
    print(f"  Chunk {i}: [{count_tokens(chunk)} tokens] {preview}...")

## 3.1 Contextual Enrichment

> 📚 **TERM: Contextual Retrieval**
> A technique introduced by Anthropic where an LLM reads the full document alongside each chunk and generates a short context summary (2-3 sentences) that is prepended to the chunk before embedding. This gives the embedding model the context it needs to create a more specific, accurate vector representation.

Here's the problem. Consider this chunk from our executive order:

> *"The committee shall submit a report by September 2024 that identifies potential generative AI initiatives pertinent to agency operations."*

Without context, an embedding model doesn't know:
- **Which** committee? (WaTech, in collaboration with cabinet agencies)
- **What** report? (A feasibility assessment of AI integration)
- **What** document? (Washington State Executive Order 24-01 on AI)

Contextual enrichment fixes this by prepending a summary:

> *"This chunk is from Washington State Executive Order 24-01 on Artificial Intelligence, signed by Governor Jay Inslee in January 2024. It describes a directive to WaTech regarding AI initiative reporting."*
>
> *"The committee shall submit a report by September 2024..."*

The enriched chunk embeds to a much more specific vector — and retrieval improves dramatically.

In [ ]:
class ChunkContext(BaseModel):
    """A concise context summary for a document chunk."""
    context: str = Field(
        description="A 2-3 sentence summary providing context for this chunk within the larger document"
    )


def _context_messages(chunk: str, document: str) -> list[dict]:
    """Build the messages list for contextual enrichment."""
    return [
        {
            "role": "system",
            "content": (
                "You are an expert at situating document chunks within their source document. "
                "Generate a concise 2-3 sentence context that explains what this chunk is about "
                "and where it fits within the larger document. Focus on information that would "
                "help a search engine understand the chunk's topic and relevance."
            ),
        },
        {
            "role": "user",
            "content": f"<document>\n{document}\n</document>\n\n"
            f"<chunk>\n{chunk}\n</chunk>\n\n"
            "Provide a short context summary for this chunk.",
        },
    ]


def add_context(chunk: str, document: str, model: str = "gpt-4o-mini") -> ChunkContext:
    """Generate a context summary for a chunk using the full document as reference."""
    response = litellm.completion(
        model=model,
        messages=_context_messages(chunk, document),
        response_format=ChunkContext,
    )
    return ChunkContext.model_validate_json(response.choices[0].message.content)

In [ ]:
# Enrich 3 chunks and show before/after
demo_indices = [0, 5, min(10, len(chunks) - 1)]

for i in demo_indices:
    chunk = chunks[i]
    result = add_context(chunk, pdf_text)

    print(f"=== Chunk {i} ({count_tokens(chunk)} tokens) ===")
    print(f"Original:  {chunk.replace(chr(10), ' ')[:120]}...")
    print(f"Context:   {result.context}")
    print(f"Combined:  {count_tokens(result.context)} + {count_tokens(chunk)} = {count_tokens(result.context + ' ' + chunk)} tokens")
    print()

In [ ]:
import time


async def add_context_async(chunk: str, document: str, model: str = "gpt-4o-mini") -> ChunkContext:
    """Async version of add_context — enables concurrent LLM calls."""
    response = await litellm.acompletion(
        model=model,
        messages=_context_messages(chunk, document),
        response_format=ChunkContext,
    )
    return ChunkContext.model_validate_json(response.choices[0].message.content)


# Enrich first 8 chunks concurrently
batch_size = 8
batch_chunks = chunks[:batch_size]

start = time.time()
tasks = [add_context_async(chunk, pdf_text) for chunk in batch_chunks]
context_results = await asyncio.gather(*tasks)
elapsed = time.time() - start

print(f"Enriched {len(context_results)} chunks in {elapsed:.1f}s (async)\n")
for i, result in enumerate(context_results):
    print(f"  Chunk {i}: {result.context[:100]}...")

In [ ]:
# Show token overhead from contextual enrichment
print(f"{'Chunk':>6s}  {'Original':>10s}  {'Context':>10s}  {'Combined':>10s}  {'Overhead':>10s}")
print("-" * 55)

enriched_data = []
for i, (chunk, ctx) in enumerate(zip(batch_chunks, context_results)):
    orig_tokens = count_tokens(chunk)
    ctx_tokens = count_tokens(ctx.context)
    combined = ctx.context + "\n---\n" + chunk
    combined_tokens = count_tokens(combined)
    overhead = combined_tokens - orig_tokens

    print(f"{i:6d}  {orig_tokens:10d}  {ctx_tokens:10d}  {combined_tokens:10d}  +{overhead:9d}")

    enriched_data.append({
        "chunk_index": i,
        "original": chunk,
        "context": ctx.context,
        "contextualized": combined,
        "original_tokens": orig_tokens,
        "combined_tokens": combined_tokens,
    })

# Save results
output_path = TEMP / "enriched-contextual.json"
output_path.write_text(json.dumps(enriched_data, indent=2))
print(f"\nSaved to {output_path}")

### Contextual Enrichment: Results and Cost

Each chunk gained ~65-105 tokens of context — roughly 30-50% overhead. That's a small price for much better embeddings.

**Cost analysis** (gpt-4o-mini pricing):
- Input: ~2,500 tokens per call (document + chunk + prompt) × $0.15/1M tokens = **~$0.0004/chunk**
- Output: ~50 tokens per call × $0.60/1M tokens = ~$0.00003/chunk
- **Total: ~$0.0004/chunk**, or **$0.40 per 1,000 chunks**
- With prompt caching (Anthropic, OpenAI): document tokens are cached after first call → **~$0.08 per 1,000 chunks**

**Anthropic's benchmark results**:
| Configuration | Retrieval Failure Reduction |
|---|---|
| Contextual embeddings alone | 35% fewer failures |
| Contextual embeddings + BM25 hybrid | **49% fewer failures** |
| Contextual embeddings + BM25 + reranking | **67% fewer failures** |

> 💡 **Tip**: Contextual enrichment is the single highest-impact enrichment technique. It's cheap, simple, and the results are dramatic. If you only do one enrichment, do this one.

## 3.2 Hypothetical Question Generation

> 📚 **TERM: Hypothetical Question Generation**
> An index-time technique where an LLM generates questions that each chunk can answer. These questions are embedded alongside (or instead of) the chunk text, so that user queries — which are typically phrased as questions — match more naturally.

Here's the **semantic gap** problem. A user types:

> *"What are the requirements for AI procurement in Washington state?"*

But the relevant chunk says:

> *"DES, in collaboration with WaTech, shall update the state's technology procurement and contract term templates, incorporating equity, bias, and algorithmic risk assessments..."*

The user's **question** and the chunk's **statement** use different vocabulary and structure. Embedding similarity may be low, even though the chunk is a perfect answer.

**Hypothetical question generation** bridges this gap:

```
Chunk: "DES shall update procurement templates..."
  ↓ LLM generates questions
  Q1: "What are the AI procurement requirements for Washington state agencies?"
  Q2: "Who is responsible for updating state technology procurement templates?"
  Q3: "What must be included in AI procurement risk assessments?"
  ↓ Embed questions + chunk → Index
```

Now when the user searches, their question matches the generated questions — which point to the right chunk.

> **Note**: This is an **index-time** technique (questions are generated once, during preprocessing). It's different from **HyDE** (Hypothetical Document Embeddings), which generates hypothetical *answers* at **query time**.

In [ ]:
class GeneratedQuestions(BaseModel):
    """Questions that a chunk of text can answer."""
    chain_of_thought: str = Field(
        description="Brief reasoning about what information this chunk contains and what questions it can answer"
    )
    questions: list[str] = Field(
        description="3-5 diverse questions that this text chunk can answer",
        min_length=3,
        max_length=5,
    )


def _question_messages(chunk: str) -> list[dict]:
    """Build the messages list for question generation."""
    return [
        {
            "role": "user",
            "content": (
                "Read this text chunk and generate 3-5 diverse questions that a user "
                "might ask where this chunk would be a good answer.\n\n"
                "Include both specific factual questions and broader conceptual ones. "
                "Use natural language — the kind of questions a real person would type "
                "into a search bar.\n\n"
                f"Text chunk:\n{chunk}"
            ),
        }
    ]


def generate_questions(chunk: str, model: str = "gpt-4o-mini") -> GeneratedQuestions:
    """Generate hypothetical questions that a chunk can answer."""
    response = litellm.completion(
        model=model,
        messages=_question_messages(chunk),
        response_format=GeneratedQuestions,
    )
    return GeneratedQuestions.model_validate_json(response.choices[0].message.content)

In [ ]:
# Generate questions for 3 chunks
demo_indices = [0, 5, min(10, len(chunks) - 1)]

for i in demo_indices:
    result = generate_questions(chunks[i])
    preview = chunks[i].replace("\n", " ")[:100]

    print(f"=== Chunk {i} ===")
    print(f"Text: {preview}...")
    print("\nGenerated questions:")
    for j, q in enumerate(result.questions):
        print(f"  {j + 1}. {q}")
    print()

# Show chain of thought for the first one
print("--- Chain of thought (chunk 0) ---")
result = generate_questions(chunks[0])
print(result.chain_of_thought)

In [ ]:
async def generate_questions_async(chunk: str, model: str = "gpt-4o-mini") -> GeneratedQuestions:
    """Async version of generate_questions."""
    response = await litellm.acompletion(
        model=model,
        messages=_question_messages(chunk),
        response_format=GeneratedQuestions,
    )
    return GeneratedQuestions.model_validate_json(response.choices[0].message.content)


# Generate questions for first 8 chunks concurrently
start = time.time()
tasks = [generate_questions_async(chunk) for chunk in batch_chunks]
question_results = await asyncio.gather(*tasks)
elapsed = time.time() - start

print(f"Generated questions for {len(question_results)} chunks in {elapsed:.1f}s\n")
print(f"{'Chunk':>6s}  {'Questions':>10s}  Sample Question")
print("-" * 80)
for i, result in enumerate(question_results):
    print(f"{i:6d}  {len(result.questions):10d}  {result.questions[0][:60]}...")

In [ ]:
# Show how questions are used at index time
i = 3  # Pick a representative chunk
print(f"=== Chunk {i} ===")
print(chunks[i][:200].replace("\n", " ") + "...\n")
print(f"Generated questions (each gets its own embedding, all pointing to chunk {i}):")
for j, q in enumerate(question_results[i].questions):
    print(f"  → {q}")

# Calculate total embeddings needed
total_questions = sum(len(r.questions) for r in question_results)
print("\n--- Embedding math ---")
print(f"Chunks: {len(batch_chunks)}")
print(f"Total questions generated: {total_questions}")
print(f"Average questions per chunk: {total_questions / len(batch_chunks):.1f}")
print(f"Total vectors to embed: {len(batch_chunks)} (chunks) + {total_questions} (questions) = {len(batch_chunks) + total_questions}")

# Save results
questions_data = []
for i, (chunk, result) in enumerate(zip(batch_chunks, question_results)):
    questions_data.append({
        "chunk_index": i,
        "chunk": chunk,
        "questions": result.questions,
        "chain_of_thought": result.chain_of_thought,
    })

output_path = TEMP / "enriched-questions.json"
output_path.write_text(json.dumps(questions_data, indent=2))
print(f"\nSaved to {output_path}")

**How many questions?** 
People usually generate 3-5 per chunk.

**Embedding separately or together?**: 
1. Separate: Embed each question independently. At query time, if a question matches, return its parent chunk. More vectors but better matching.
2. Together: Append questions to the chunk text before embedding. Fewer vectors but the questions dilute the chunk's embedding.

Option 1 is more common

**Cost**: Similar to contextual enrichment — ~$0.0004/chunk for generation + embedding costs for the extra vectors.

**Combining with contextual enrichment**: These techniques are complementary:
- **Contextual enrichment** helps the chunk's *own* embedding be more specific
- **Hypothetical questions** create *additional* embeddings that match query patterns

Many production systems use both.

> 💡 **Tip**: If your users tend to ask questions (search bars, chatbots), hypothetical question generation will have a big impact. If your system mostly does keyword-style retrieval, the benefit is smaller.

## 3.3 Metadata Extraction

> 📚 **TERM: Metadata Extraction**
> Using an LLM to extract structured information from each chunk — topics, named entities, dates, key terms, and more. This metadata is stored alongside the chunk's embedding and enables **structured filtering** on top of vector search.

Semantic search finds chunks with similar *meaning*. But sometimes you need to filter by specific *attributes*:

- *"Show me chunks about **procurement** from **2024**"*
- *"Find all directives involving **WaTech**"*
- ...

These queries combine semantic similarity with structured constraints. Metadata makes this possible.


In [ ]:
class ChunkMetadata(BaseModel):
    """Structured metadata extracted from a document chunk."""
    topics: list[str] = Field(
        description="2-4 topic labels that describe what this chunk is about (e.g., 'procurement', 'workforce training', 'risk assessment')"
    )
    entities: list[str] = Field(
        description="Named entities mentioned: organizations, people, laws, programs (e.g., 'WaTech', 'Governor Inslee', 'Executive Order 24-01')"
    )
    dates: list[str] = Field(
        description="Dates or deadlines mentioned (e.g., 'September 2024', 'January 2025')"
    )
    document_type: str = Field(
        description="The type of document this chunk comes from (e.g., 'executive order', 'policy directive', 'report')"
    )
    key_terms: list[str] = Field(
        description="3-5 important technical or domain-specific terms in the chunk"
    )
    summary: str = Field(
        description="A one-sentence summary of what this chunk says"
    )


def _metadata_messages(chunk: str) -> list[dict]:
    """Build the messages list for metadata extraction."""
    return [
        {
            "role": "user",
            "content": (
                "Extract structured metadata from this text chunk. "
                "Be specific and factual — only include information that is "
                "explicitly stated in the text.\n\n"
                f"Text chunk:\n{chunk}"
            ),
        }
    ]


def extract_metadata(chunk: str, model: str = "gpt-4o-mini") -> ChunkMetadata:
    """Extract structured metadata from a chunk using an LLM."""
    response = litellm.completion(
        model=model,
        messages=_metadata_messages(chunk),
        response_format=ChunkMetadata,
    )
    return ChunkMetadata.model_validate_json(response.choices[0].message.content)

In [ ]:
# Extract metadata for 3 chunks
demo_indices = [0, 5, min(10, len(chunks) - 1)]

for i in demo_indices:
    meta = extract_metadata(chunks[i])
    preview = chunks[i].replace("\n", " ")[:100]

    print(f"=== Chunk {i} ===")
    print(f"Text:     {preview}...")
    print(f"Topics:   {meta.topics}")
    print(f"Entities: {meta.entities}")
    print(f"Dates:    {meta.dates}")
    print(f"Type:     {meta.document_type}")
    print(f"Terms:    {meta.key_terms}")
    print(f"Summary:  {meta.summary}")
    print()

In [ ]:
async def extract_metadata_async(chunk: str, model: str = "gpt-4o-mini") -> ChunkMetadata:
    """Async version of extract_metadata."""
    response = await litellm.acompletion(
        model=model,
        messages=_metadata_messages(chunk),
        response_format=ChunkMetadata,
    )
    return ChunkMetadata.model_validate_json(response.choices[0].message.content)


# Extract metadata for first 8 chunks
start = time.time()
tasks = [extract_metadata_async(chunk) for chunk in batch_chunks]
metadata_results = await asyncio.gather(*tasks)
elapsed = time.time() - start

print(f"Extracted metadata for {len(metadata_results)} chunks in {elapsed:.1f}s\n")

# Build a DataFrame for easy viewing
rows = []
for i, (chunk, meta) in enumerate(zip(batch_chunks, metadata_results)):
    rows.append({
        "chunk": i,
        "tokens": count_tokens(chunk),
        "summary": meta.summary[:80] + "..." if len(meta.summary) > 80 else meta.summary,
        "topics": ", ".join(meta.topics),
        "entities": ", ".join(meta.entities),
        "dates": ", ".join(meta.dates),
    })

metadata_df = pd.DataFrame(rows)
metadata_df

In [ ]:
# Metadata enables structured filtering — like SQL WHERE clauses on your chunks

# 1. Find chunks about procurement
print("=== Chunks about 'procurement' ===")
procurement = metadata_df[metadata_df["topics"].str.contains("procurement", case=False, na=False)]
for _, row in procurement.iterrows():
    print(f"  Chunk {row['chunk']}: {row['summary']}")
print()

# 2. Find chunks with 2024 deadlines
print("=== Chunks mentioning 2024 deadlines ===")
has_2024 = metadata_df[metadata_df["dates"].str.contains("2024", na=False)]
for _, row in has_2024.iterrows():
    dates = row["dates"]
    print(f"  Chunk {row['chunk']}: {row['summary']} [{dates}]")
print()

# 3. Find chunks mentioning WaTech
print("=== Chunks mentioning WaTech ===")
watech = metadata_df[metadata_df["entities"].str.contains("WaTech", case=False, na=False)]
for _, row in watech.iterrows():
    print(f"  Chunk {row['chunk']}: {row['summary']}")

In [ ]:
# Save enriched metadata as JSON
metadata_json = []
for i, (chunk, meta) in enumerate(zip(batch_chunks, metadata_results)):
    metadata_json.append({
        "chunk_index": i,
        "chunk": chunk,
        "topics": meta.topics,
        "entities": meta.entities,
        "dates": meta.dates,
        "document_type": meta.document_type,
        "key_terms": meta.key_terms,
        "summary": meta.summary,
    })

output_path = TEMP / "enriched-metadata.json"
output_path.write_text(json.dumps(metadata_json, indent=2))
print(f"Saved to {output_path}")

And here is how to do this with LanceDB:

In [ ]:
import lancedb
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector

# Same embedding function from Lecture 6
embedding_func = get_registry().get("openai").create(name="text-embedding-3-small")


# Define schema with metadata columns — just like Lecture 6, but with extra fields
class EnrichedChunk(LanceModel):
    chunk_index: int
    content: str = embedding_func.SourceField()  # auto-embedded by LanceDB
    vector: Vector(embedding_func.ndims()) = embedding_func.VectorField()
    topics: str  # "procurement, AI governance"
    entities: str  # "WaTech, DES"
    dates: str  # "September 2024, January 2025"
    document_type: str
    summary: str


# Prepare data from our metadata extraction results
data = [
    {
        "chunk_index": i,
        "content": chunk,
        "topics": ", ".join(meta.topics),
        "entities": ", ".join(meta.entities),
        "dates": ", ".join(meta.dates),
        "document_type": meta.document_type,
        "summary": meta.summary,
    }
    for i, (chunk, meta) in enumerate(zip(batch_chunks, metadata_results))
]

# Create a LanceDB table — embeddings are computed automatically
db = lancedb.connect("../temp/lancedb-preprocessing")
table = db.create_table("enriched_chunks", schema=EnrichedChunk, mode="overwrite")
table.add(data)
table.create_fts_index("content", replace=True)

print(f"Created table with {table.count_rows()} enriched chunks")
print(f"Columns: {[f.name for f in table.schema]}")


In [ ]:
# --- Unfiltered vector search ---
query = "What are the procurement requirements for AI?"
unfiltered = table.search(query).limit(3).to_list()

print(f'Query: "{query}"')
print("No filter — top 3 results:\n")
for r in unfiltered:
    print(f"  Chunk {r['chunk_index']}: [{r['topics']}]")
    print(f"    {r['summary']}")
    print()

# --- Filtered: only chunks about procurement ---
filtered = (
    table.search(query)
    .where("topics LIKE '%procurement%'", prefilter=True)
    .limit(3)
    .to_list()
)

print("Same query + filter: topics LIKE '%procurement%'")
print(f"Filtered — {len(filtered)} result(s):\n")
for r in filtered:
    print(f"  Chunk {r['chunk_index']}: [{r['topics']}]")
    print(f"    {r['summary']}")
    print()

# --- Filtered: 2024 deadlines ---
query2 = "training and workforce development requirements"
filtered_2024 = (
    table.search(query2)
    .where("dates LIKE '%2024%'", prefilter=True)
    .limit(3)
    .to_list()
)

print(f'Query: "{query2}"')
print(f"Filter: dates LIKE '%2024%' — {len(filtered_2024)} result(s):\n")
for r in filtered_2024:
    print(f"  Chunk {r['chunk_index']}: [{r['dates']}] {r['summary']}")
    print()

# --- Filtered: specific entity ---
query3 = "guidelines and assessment frameworks"
filtered_entity = (
    table.search(query3)
    .where("entities LIKE '%WaTech%'", prefilter=True)
    .limit(3)
    .to_list()
)

print(f'Query: "{query3}"')
print(f"Filter: entities LIKE '%WaTech%' — {len(filtered_entity)} result(s):\n")
for r in filtered_entity:
    print(f"  Chunk {r['chunk_index']}: [{r['entities']}] {r['summary']}")


### Metadata Filtering in Practice

We just saw **filtered vector search** in action — combining embedding similarity with SQL-like metadata constraints. This is the same LanceDB pattern from Lecture 6, extended with metadata columns.

The `.where()` syntax supports standard SQL expressions:

| Expression | Example | Use Case |
|---|---|---|
| Text matching | `topics LIKE '%procurement%'` | Find chunks about a topic |
| Exact match | `document_type = 'executive order'` | Filter by category |
| Numeric comparison | `chunk_index > 5` | Positional filtering |
| Combined | `entities LIKE '%WaTech%' AND dates LIKE '%2024%'` | Multi-field filtering |

The `prefilter=True` parameter filters **before** vector search (narrows the search space), which is faster and ensures all returned results match your constraints.

**When to use metadata filtering**:
- Your corpus spans multiple document types, time periods, or topics
- Users combine content queries with structural constraints (*"AI procurement rules from 2024"*)
- You need access control (filter by department, classification level)
- You want faceted search (browse by topic, date, entity)

> 💡 **Tip**: Even if you don't need filtering today, extracting metadata at indexing time is cheap insurance. It's much easier to add a filter later than to re-process your entire corpus.

# Resources
---

### Preprocessing
- [markdownify](https://github.com/matthewwithanm/python-markdownify) — HTML to Markdown
- [python-docx](https://python-docx.readthedocs.io) — Read/write Word documents
- [pymupdf4llm](https://pymupdf.readthedocs.io/en/latest/rag.html) — Fast PDF to Markdown
- [Docling](https://github.com/docling-project/docling) (IBM) — AI-powered document conversion
- [MinerU](https://github.com/opendatalab/MinerU) (OpenDataLab) — AI PDF parser
- [Marker](https://github.com/datalab-to/marker) (Datalab) — Fast ML-based PDF conversion
- [MarkItDown](https://github.com/microsoft/markitdown) (Microsoft) — Multi-format converter
- [pdfplumber](https://github.com/jsvine/pdfplumber) — Precise table extraction from PDFs
- [trafilatura](https://trafilatura.readthedocs.io) — Web content extraction
- [LlamaParse](https://docs.cloud.llamaindex.ai/llamaparse/getting_started) — Commercial PDF parsing
- [Unstructured](https://docs.unstructured.io) — Commercial document ETL

### Chunking
- [Evaluating Chunking Strategies for Retrieval](https://research.trychroma.com/evaluating-chunking) — Chroma's benchmark study
- [Five Levels of Chunking](https://www.youtube.com/watch?v=8OJC21T2SL4) — Greg Kamradt's influential video
- [Chunking Strategies for LLM Applications](https://www.pinecone.io/learn/chunking-strategies/) — Pinecone's guide
- [Finding the Best Chunking Strategy](https://developer.nvidia.com/blog/finding-the-best-chunking-strategy-for-accurate-ai-responses/) — NVIDIA's benchmark
- [Late Chunking](https://jina.ai/news/late-chunking-in-long-context-embedding-models/) — Jina AI's context-aware embeddings
- [LangChain Text Splitters](https://python.langchain.com/docs/how_to/#text-splitters) — RecursiveCharacterTextSplitter and more
- [LlamaIndex Node Parsers](https://docs.llamaindex.ai/en/stable/module_guides/loading/node_parsers/) — Chunking within LlamaIndex
- [Chonkie](https://docs.chonkie.ai/) — Lightweight chunking library
- [tiktoken](https://github.com/openai/tiktoken) — OpenAI's fast tokenizer

### Enrichment
- [Contextual Retrieval](https://www.anthropic.com/news/contextual-retrieval) — Anthropic's approach to enriching chunks with document context
- [Hypothetical Document Embeddings (HyDE)](https://arxiv.org/abs/2212.10496) — Query-time question generation (related concept)
- [LlamaIndex Metadata Extractors](https://docs.llamaindex.ai/en/stable/module_guides/loading/documents_and_nodes/usage_metadata_extractor/) — Built-in extractors for questions, summaries, entities
- [Instructor](https://python.useinstructor.com/) — Structured LLM output with Pydantic validation
- [LiteLLM](https://docs.litellm.ai/) — Unified LLM API (what we use in this course)
- [LanceDB Filtering](https://lancedb.github.io/lancedb/sql/) — SQL-like filtering on vector database metadata
- [Pydantic](https://docs.pydantic.dev/) — Data validation and structured output models

# Homework 6: Multimodal Image Search

---

Time to put today's concepts into practice! Your homework applies preprocessing and enrichment to a **multimodal** setting: building a searchable index over images instead of text documents.